# 레지스트리·다이제스트와 클라우드 배포

태그와 다이제스트가 가리키는 배포 대상을 비교합니다.

## 코드 읽기와 개념 경계

이미지 태그는 사람이 읽기 좋은 이동 가능한 이름이고, 다이제스트는 이미지 내용으로 결정되는 식별자입니다. 같은 `latest`라는 태그도 시간이 지나면 다른 이미지를 가리킬 수 있지만 `sha256:...` 다이제스트는 특정 바이트를 가리킵니다. 실습은 v1과 v2 이미지를 빌드해 서로 다른 식별자를 확인하고 배포 기록에 v1 다이제스트를 저장합니다.

다이제스트 고정은 무엇을 배포했는지 재현하는 데 필요하지만 그 이미지가 안전하거나 올바르다는 보장은 아닙니다. 서명·취약점 검사·테스트 결과를 같은 산출물과 연결해야 합니다. 클라우드 배포 권한과 환경 승인은 이미지 생성 권한과 분리하여, CI가 임의의 운영 대상에 바로 쓰지 못하게 합니다.

In [1]:
from pathlib import Path
import hashlib, subprocess, sys, tempfile, json
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()

In [2]:
for revision in ("v1", "v2"):
    built = subprocess.run(["docker", "build", "-q", "-t", f"ch33-demo:{revision}", "--build-arg", f"APP_REVISION={revision}", "."], cwd=lab, text=True, capture_output=True)
    print(revision, built.stdout.strip()[:27], "exit_code=", built.returncode)
    assert built.returncode == 0
digest_v1 = subprocess.check_output(["docker", "image", "inspect", "ch33-demo:v1", "--format", "{{.Id}}"], text=True).strip()
digest_v2 = subprocess.check_output(["docker", "image", "inspect", "ch33-demo:v2", "--format", "{{.Id}}"], text=True).strip()
print("same_tag=latest", "same_digest=", digest_v1 == digest_v2)
assert digest_v1 != digest_v2

v1 sha256:1cb76db0ea58f43f50dc exit_code= 0


v2 sha256:4067958c02de6da69345 exit_code= 0
same_tag=latest same_digest= False


In [3]:
with tempfile.TemporaryDirectory() as temp:
    t = Path(temp)
    (t / "scripts").mkdir()
    (t / "runs").mkdir()
    (t / "scripts/deploy.py").write_text((lab / "scripts/deploy.py").read_text())
    result = subprocess.run([sys.executable, "scripts/deploy.py", digest_v1], cwd=t, text=True, capture_output=True)
    print(result.stdout.strip(), "exit_code=", result.returncode)
    print((t / "runs/deployment.json").read_text().strip())
    assert result.returncode == 0

deployed revision=1 digest=sha256:1cb76db0ea58... exit_code= 0
[
  {
    "revision": 1,
    "event": "deploy",
    "digest": "sha256:1cb76db0ea58f43f50dc84141dfc2896efed098aa2e67e01fc9f5116cc417328",
    "status": "ready"
  }
]


## 결과 해석

같은 latest 태그라는 설명만으로 실제 바이트를 식별할 수 없지만, 내용에서 계산한 다이제스트는 v1과 v2가 달랐습니다. 로컬 배포 기록은 다이제스트를 저장했습니다.